In [ ]:
!pip install transformers accelerate torch safetensors numpy scipy openai

!pip install -U bitsandbytes

In [ ]:
import os
import pandas as pd
import openai
from tqdm import tqdm

openai.api_key = "OPENAI_API_KEY"

SIM_XLSX_PATH = "/path/to/simulations"
PROMPT_LEVELS = {
    "Level 1": "Generate a list of 10 distinct student activities based on the provided questions, relationships, and variables so that the activities follow a natural learning order to incrementally develop understanding.",
    "Level 2": "Using the provided questions, relationships, and variables, generate a structured list of 10 distinct student activities. The activities should select one key variable to vary while keeping the others constant and students should record outputs for each variation. The activities should use outcomes from earlier activities as input for later ones so that they follow a natural learning order to incrementally develop understanding.",
    "Level 3": "Based on the provided questions, relationships, and variables, generate a detailed list of 10 distinct student activities by following these steps:\nSelect one variable to vary while keeping others constant and record the corresponding outputs.\nActivities should use outcomes from earlier activities as input for later ones.\nGroup activities that build upon each other based on overlapping variables or outputs.\nProvide clear instructions emphasizing data collection and recording observations.\nActivities should follow a natural learning order to incrementally develop understanding.",
    "Level 4": "Based on the provided questions, relationships, and variables, generate a detailed list of 10 distinct student activities by following these steps:\nSelect one variable to vary while keeping others constant and record the corresponding outputs.\nActivities should use outcomes from earlier activities as input for later ones.\nGroup activities that build upon each other based on overlapping variables or outputs.\nProvide clear instructions emphasizing data collection and recording observations.\nActivities should follow a natural learning order to incrementally develop understanding. For example, given the projectile simulation, here are a few questions: \nExplain how the launch angle affects the range of the projectile. \nWhich of the following factors can affect the range of the projectile? \nDoes increasing the initial velocity of a projectile always increase its maximum height? \nBased on these questions, here are some probable activities: \nExperiment with varying the initial velocity while keeping the launch angle constant. Record your observations on how it affected the range and time of flight. \nObserve and describe the impact of acceleration due to gravity on the time of flight. \nCompare and contrast the trajectory of a projectile at different launch angles with the same initial velocity. \nDo not generate these examples of activities if they’re irrelevant. Generate activities given the provided variables, relationships, and questions based on the relevant simulation only."
}

df = pd.read_excel(SIM_XLSX_PATH)

def call_gpt4(prompt):
    try:
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are an educational assistant that generates activities for students."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=768,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

for level_name, level_prompt in PROMPT_LEVELS.items():
    output_col = f"gpt-4 - {level_name}"
    df[output_col] = ""

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Generating {level_name}"):
        full_prompt = (
            f"{level_prompt}\n\n"
            f"Questions: {row['Questions']}\n"
            f"Relationships: {row['Relationships']}\n"
            f"Variables: {row['Variables']}\n"
        )
        result = call_gpt4(full_prompt)
        df.at[idx, output_col] = result

df.to_excel("generated_activities_gpt4.xlsx", index=False)
print("✅ Activity generation complete for GPT-4. Saved to 'generated_activities_gpt4.xlsx'")